In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/tosinforlly@gmail.com/fmcg_project/1_setup/utilities

In [0]:
dbutils.widgets.text('catalog', 'spotizone', 'Catalog')
dbutils.widgets.text('data_source', 'gross_price', 'Data Source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

storage_path = f's3://sport-bar/{data_source}/*.csv'

##### BRONZE LAYER

In [0]:
df_gp = (
    spark.read.format("csv")
    .option("inferSchema", True)
    .option("header", True)
    .load(storage_path)
    .withColumn("ingest_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name")
)

###### Write: Into Bronze

In [0]:
df_gp.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

#### SILVER LAYER

In [0]:
df_gp = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source}')

In [0]:
df_silver = df_gp
display(df_silver)

In [0]:
df_silver.printSchema()

##### Tranformation

In [0]:
# Checking for duplicates in primary key 
display(df_silver.count())

- 1. Resolve Inconsistent Data Format 

In [0]:
# Date Format Lit
date_format = ['yyyy/MM/dd', 'dd/MM/yyyy', 'yyyy-MM-dd', 'dd-MM-yyyy']

# Parse Date Formats
df_silver = (
    df_silver\
        .withColumn(
            'month', F.coalesce(
                F.try_to_date(F.col('month'), 'yyyy/MM/dd'),
                F.try_to_date(F.col('month'), 'dd/MM/yyyy'),
                F.try_to_date(F.col('month'), 'yyyy-MM-dd'),
                F.try_to_date(F.col('month'), 'dd-MM-yyyy')
            )
        )
)

In [0]:
# Now check date format consitency
display(df_silver.select('month').limit(5))

- 2. Handling gross_price Value

In [0]:
display(df_silver.limit(10))

In [0]:
df_silver = (
    df_silver
        .withColumn(
            'gross_price',
            F.when(
                F.col('gross_price').rlike("^-?[0-9]+(\\.[0-9]+)?$"),
                F.abs(F.col('gross_price').cast('double'))
            ).otherwise(0)
        )
)

In [0]:
display(df_silver.limit(5))

- 3. Enrich With product_code From Silver

In [0]:
# Call Products table in silver layer
df_prod = spark.table('spotizone.silver.products')

# Join gross_price with product table
df_silver = df_silver.join(
    df_prod.select(['product_id', 'product_code']),
    on='product_id',
    how='inner'
)

# Select Required Columns
df_silver = df_silver.select('product_id','product_code', 'month', 'gross_price', 'ingest_timestamp', 'file_name')


In [0]:
display(df_silver.limit(5))

##### Write Into Silver Layer

In [0]:
df_silver.write\
    .mode('overwrite')\
    .format('delta')\
    .option('enableChangeDataFeed','true')\
    .option('mergeSchema','true')\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

#### GOLD LAYER

In [0]:
df_silver = spark.table(f'{catalog}.{silver_schema}.{data_source}')

In [0]:
df_gold = df_silver.select("product_code", "month", "gross_price")
display(df_gold.limit(5))

In [0]:
df_gold.write\
    .mode('overwrite')\
    .format('delta')\
    .option('enableChangeDataFeed','true')\
    .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

##### Merge With Parent Company

In [0]:
df_gold = spark.table(f'{catalog}.{gold_schema}.sb_dim_{data_source}')
display(df_gold.limit(10))

In [0]:
# Create year and is_zero flag in sb_dim_products
df_merge = df_gold\
    .withColumn('year', F.year('month'))\
    .withColumn('is_zero', F.when(F.col('gross_price') == 0, 1).otherwise(0))
display(df_merge.limit(5))

##### Using Window function, Rank products by latest month

In [0]:
from pyspark.sql.window import Window

In [0]:
windows = Window\
    .partitionBy('product_code', 'year')\
    .orderBy(F.col('is_zero'), F.col('month').desc())

# Ranking
df_merge = df_merge.withColumn('rank', F.row_number().over(windows)).filter(F.col('rank') == 1)

In [0]:
display(df_merge.limit(5))

In [0]:
## elect required cols

df_merge = df_merge.select("product_code", "gross_price", "year").withColumnRenamed("gross_price", "price_inr")

# change year to string
df_merge = df_merge.withColumn("year", F.col("year").cast("string"))

display(df_merge.limit(5))

In [0]:
%sql
CONVERT TO DELTA spotizone.gold.dim_gross_price

In [0]:

# Merge with Parent company 
target = DeltaTable.forName(spark, "spotizone.gold.dim_gross_price").alias("target")

target.merge(
    source=df_merge.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()